In [1]:

from collections import deque
from agent import Agent
from tqdm import trange
import math
import argparse
import os
import torch
from newenv import TicTacToeEnv
buffer = deque(maxlen=25000)

env = TicTacToeEnv(render=False)
agent = Agent()

rewards, loss = [],[]
transition_reward = 0
state,info = env.reset()

epochs = 250000
os.makedirs("Outputs/Results",exist_ok=True)

In [2]:
for i in trange(epochs):
    eps = 0.05 + 0.95 * math.exp(-i / (epochs/5))   # correct pacing for 200k epochs   # for 200,000 epochs
    state_before = state
    player1_action = agent.act_e_greedyy(state,eps)
    next_state, reward, terminated, truncated, info = env.step(player1_action)

    done = terminated or truncated
 
    if done:
            transition_reward = reward
            buffer.append((state,player1_action,reward,next_state,done))
            state,info = env.reset()


    else:

        player2_action = agent.act_e_greedyy(next_state,eps)

        final_state, player2_reward, terminated, truncated, info = env.step(player2_action)
        done = terminated or truncated

        buffer.append((next_state,player2_action,player2_reward,final_state,done))
        buffer.append((state_before, player1_action, reward, next_state, False))


        state = final_state

        if done:
             state,info = env.reset()

        
    if i > 100:
        loss.append(agent.train_iter(buffer))

    if i  > 100 and i % 1000 == 0:
        agent.update_target_net()


torch.save(
    agent.online_net.state_dict(),
    "Outputs/Results/testmine.pt"
)

100%|██████████| 250000/250000 [11:04<00:00, 376.14it/s]
